# gas-sim-pro · Training Notebook
**Always open fresh from GitHub** — do not reuse old Colab tabs.

Run all cells top to bottom. Do not skip cells.
Training completes in ~10 minutes on Colab free tier.

In [ ]:
# ── Cell 1 — Config + auth + registry ───────────────────────────────────
PROJECT_ID = 'gas-sim-pro-ii'
BUCKET     = f'{PROJECT_ID}-gas-sim-data'

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
import json

gcs    = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

def read_registry():
    return json.loads(bucket.blob('model_registry.json').download_as_text())

reg = read_registry()

print('── Registry ──────────────────────────────')
print(f'  last_trained:     {reg["last_trained"]}')
print(f'  last_data_upload: {reg["last_data_upload"]}')
print(f'  feature_version:  {reg["feature_version"]}')
print(f'  current MAE:      {reg["mae"]}')
print(f'  previous MAE:     {reg["previous_mae"]}')
print(f'  rows_trained_on:  {reg["rows_trained_on"]}')
print(f'  gate_status:      {reg["gate_status"]}')
print('──────────────────────────────────────────')
print('Cell 1 complete.')


In [ ]:
# ── Cell 2 — Install dependencies ───────────────────────────────────────
!pip install -q xgboost wandb pyarrow pandas-gbq
print('Dependencies installed.')


In [ ]:
# ── Cell 3 — Load features from GCS ─────────────────────────────────────
import pandas as pd

blobs = list(gcs.bucket(BUCKET).list_blobs(prefix='features/latest/'))
paths = [f'gs://{BUCKET}/{b.name}' for b in blobs if b.name.endswith('.parquet')]
print(f'Found {len(paths)} parquet file(s)')

assert len(paths) > 0, 'No Parquet files found — run cloud.sh option 8 first'

df = pd.concat(
    [pd.read_parquet(p, storage_options={'token': 'google_default'}) for p in paths],
    ignore_index=True
)

assert len(df) >= 500, f'Only {len(df)} rows — need at least 500 to train'
print(f'Loaded {len(df):,} rows · {len(df.columns)} columns')
df[['sensor_delta','centroid_row','centroid_col','wind_angle','leak_row','leak_col']].describe()


In [ ]:
# ── Cell 4 — Feature matrix + train/val split ────────────────────────────
from sklearn.model_selection import train_test_split
import numpy as np

FEATURES = [
    'sensor_delta', 'sensor_mean', 'reading_variance',
    'centroid_row', 'centroid_col', 'coverage_ratio',
    'wind_angle', 'wind_magnitude', 'distance_to_boundary',
    'wind_x', 'wind_y', 'diffusion_rate', 'decay_factor',
    'leak_injection', 'sensor_count'
]
TARGETS = ['leak_row', 'leak_col']

X = df[FEATURES].values.astype(np.float32)
y = df[TARGETS].values.astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42
)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')


In [ ]:
# ── Cell 5 — W&B init (optional, never blocks) ───────────────────────────
import wandb

WANDB_ACTIVE = False
run = None

try:
    run = wandb.init(
        project='gas-sim-pro',
        anonymous='allow',
        mode='offline',
        config={
            'model': 'xgboost',
            'features': FEATURES,
            'n_rows': len(df),
            'feature_version': reg['feature_version'],
        }
    )
    WANDB_ACTIVE = True
    print(f'W&B run initialised (offline mode)')
except Exception as e:
    print(f'W&B unavailable ({e}) — continuing without tracking')


In [ ]:
# ── Cell 6 — Train XGBoost ───────────────────────────────────────────────
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

params = dict(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)
model = MultiOutputRegressor(xgb.XGBRegressor(**params))
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
mae    = float(np.mean(np.abs(y_pred - y_val)))
print(f'Val MAE: {mae:.4f} cells')

if WANDB_ACTIVE and run:
    wandb.log({'val_mae': mae})


In [ ]:
# ── Cell 7 — MAE gate (advisory, never stops execution) ──────────────────
# Re-read registry fresh in case it changed since Cell 1
reg = read_registry()
prod_mae = reg.get('mae') or float('inf')
prev_mae = reg.get('previous_mae')

print('── MAE Gate Assessment ──────────────────────────────────────')
print(f'  New model MAE  : {mae:.4f} cells')
print(f'  Current prod   : {prod_mae:.4f} cells')
print(f'  Previous prod  : {prev_mae}')
print(f'  Gate threshold : prod x 0.98 = {prod_mae * 0.98:.4f}')
print()

if prod_mae == float('inf'):
    GATE_STATUS = 'first_model'
    print('✅ GATE: First model — no baseline to beat. Will deploy.')

elif mae < prod_mae * 0.98:
    GATE_STATUS = 'passed'
    improvement = (prod_mae - mae) / prod_mae * 100
    print(f'✅ GATE PASSED: {improvement:.1f}% improvement. Model will be deployed.')

elif mae < prod_mae:
    GATE_STATUS = 'marginal'
    diff = (prod_mae - mae) / prod_mae * 100
    print(f'⚠️  GATE MARGINAL: {diff:.1f}% better but below the 2% threshold.')
    print('   Model exported but deploy function will likely block it.')
    print('   Consider: generate more data and retrain.')

else:
    GATE_STATUS = 'failed'
    regressed = (mae - prod_mae) / prod_mae * 100
    print(f'❌ GATE FAILED: New model is {regressed:.1f}% WORSE than production.')
    print(f'   prod MAE: {prod_mae:.4f}  new MAE: {mae:.4f}')
    print()
    print('   Possible causes:')
    print('   • Same dataset trained twice — run cloud.sh option 8 first')
    print('   • Not enough new data — generate more rows')
    print('   • Feature distribution changed — check dbt export')
    print()
    print('   Cells 8-9 will be skipped. Registry will record gate_status=failed.')
    print('   To reset and force deploy: cloud.sh option 5 then rerun notebook.')

if WANDB_ACTIVE and run:
    wandb.log({'gate_status': GATE_STATUS, 'prod_mae': prod_mae})

print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 8 — Export model ────────────────────────────────────────────────
import os, datetime, joblib

if GATE_STATUS not in ('passed', 'first_model'):
    print(f'⏭  Cell 8 skipped — gate status is [{GATE_STATUS}]')
    print('   Model will not be exported or uploaded.')
    VERSION = None
else:
    RUN_ID  = run.id if WANDB_ACTIVE and run else datetime.datetime.now().strftime('%H%M%S')
    VERSION = f"v{datetime.date.today().strftime('%Y%m%d')}-{RUN_ID}"
    os.makedirs(f'/tmp/{VERSION}', exist_ok=True)
    joblib.dump(model, f'/tmp/{VERSION}/model.joblib')
    print(f'✓ Model exported: {VERSION}')


In [ ]:
# ── Cell 9 — Push to GCS ─────────────────────────────────────────────────
if GATE_STATUS not in ('passed', 'first_model') or VERSION is None:
    print(f'⏭  Cell 9 skipped — gate status is [{GATE_STATUS}]')
else:
    blob = bucket.blob(f'models/{VERSION}/model.joblib')
    blob.upload_from_filename(f'/tmp/{VERSION}/model.joblib')
    print(f'✓ Uploaded: models/{VERSION}/model.joblib')

    if WANDB_ACTIVE and run:
        try:
            wandb.log_artifact(f'/tmp/{VERSION}/model.joblib', name='model-joblib', type='model')
        except Exception as e:
            print(f'  W&B artifact log skipped: {e}')


In [ ]:
# ── Cell 10 — Update registry ────────────────────────────────────────────
import datetime as dt

prev_version = reg.get('latest_version')
prev_mae_val = reg.get('mae')

if GATE_STATUS in ('passed', 'first_model') and VERSION is not None:
    # Gate passed — update prod model
    reg.update({
        'latest_version':   VERSION,
        'previous_version': prev_version,
        'last_trained':     dt.datetime.now(dt.timezone.utc).isoformat(),
        'model_path':       f'models/{VERSION}/model.joblib',
        'joblib_path':      f'models/{VERSION}/model.joblib',
        'mae':              mae,
        'previous_mae':     prev_mae_val,
        'rows_trained_on':  len(df),
        'feature_version':  reg['feature_version'],
        'gate_status':      GATE_STATUS,
    })
else:
    # Gate failed — record training attempt, leave prod model unchanged
    reg.update({
        'last_trained':    dt.datetime.now(dt.timezone.utc).isoformat(),
        'gate_status':     GATE_STATUS,
        'rows_trained_on': len(df),
    })

reg_blob = bucket.blob('model_registry.json')
reg_blob.upload_from_string(json.dumps(reg, indent=2), content_type='application/json')
reg_blob.cache_control = 'no-cache, no-store, max-age=0'
reg_blob.patch()

# Write last_training_result.json for TRAIN button UI feedback
result = {
    'status':     GATE_STATUS,
    'mae':        mae,
    'prod_mae':   prod_mae,
    'version':    VERSION,
    'trained_at': dt.datetime.now(dt.timezone.utc).isoformat(),
    'rows':       len(df),
}
r_blob = bucket.blob('registry/last_training_result.json')
r_blob.upload_from_string(json.dumps(result, indent=2), content_type='application/json')
r_blob.cache_control = 'no-cache, no-store, max-age=0'
r_blob.patch()

if WANDB_ACTIVE and run:
    wandb.finish()

# ── Final summary ─────────────────────────────────────────────────────────
print()
print('── Training Summary ─────────────────────────────────────────')
print(f'  Gate status  : {GATE_STATUS}')
print(f'  New MAE      : {mae:.4f}')
print(f'  Prod MAE     : {prod_mae:.4f}')
print(f'  Version      : {VERSION}')
print(f'  Rows trained : {len(df):,}')
print()
if GATE_STATUS in ('passed', 'first_model'):
    print('🚀 Deploy function will trigger automatically (~5 min).')
    print('   TRAIN button will flash green when deployed.')
elif GATE_STATUS == 'marginal':
    print('⚠️  Model slightly better but below threshold.')
    print('   Generate more data and retrain.')
    print('   TRAIN button will flash yellow.')
else:
    print('❌ Model did not improve. Prod model unchanged.')
    print('   1. cloud.sh option 8 (refresh features)')
    print('   2. Generate more data')
    print('   3. Rerun this notebook')
    print('   TRAIN button will flash yellow.')
print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 11 — Verify predictions ─────────────────────────────────────────
if GATE_STATUS not in ('passed', 'first_model') or VERSION is None:
    print(f'⏭  Cell 11 skipped — no model exported (gate: {GATE_STATUS})')
else:
    skl_pred = model.predict(X_val[:5])
    print('Predictions (first 5 rows):')
    for i in range(5):
        print(f'  leak_row={skl_pred[i][0]:.1f}  leak_col={skl_pred[i][1]:.1f}')
    print('Verification complete.')
